# Multi-Dataset Pipeline: ROI-Cropped → SAM-Med3D → TabPFN

This notebook implements a complete end-to-end pipeline for **multiple datasets** (GIST and LIPO):

1. **Preprocess** images with ROI cropping (tumor-centered volumes)
2. **Load** SAM-Med3D model via medim
3. **Extract** embeddings from the encoder
4. **Pool** embeddings to feature vectors (Global Average Pooling)
5. **Load** labels from sheet.csv
6. **Classify** with TabPFN
7. **Summarize** performance across all datasets

**Key Feature**: Uses existing functions from the codebase instead of custom implementations!

## 1. Imports and Setup

In [1]:
import sys
from pathlib import Path
import numpy as np
import torch
import SimpleITK as sitk
import torchio as tio
import medim

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import existing functions from codebase
from med3pipe.data.prepare import (
    Sam3DPaths, find_default_sam3d_root, prepare_for_sam3d_roi_cropped,
    split_validation
)
from med3pipe.sam.core import (
    build_sam3d_model, extract_embeddings_train_val, load_pooled_features,
    load_labels_from_sheet, build_y, default_feature_dirs
)

# Define paths
SAM3D_ROOT = find_default_sam3d_root(project_root)
CHECKPOINT_PATH = SAM3D_ROOT / "ckpt" / "sam_med3d_turbo.pth"

# Configuration
TARGET_SIZE = 128
ROI_MARGIN = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Datasets to process (4 datasets: GIST, LIPO, DESMOID, CRLM)
# NOTE: case_suffix must match actual directory naming (e.g., GIST-001_CT, Desmoid-001_MR)
DATASETS = [
    {
        "name": "GIST",
        "category": "gist",
        "ct_name": "ct_GIST_roi",
        "dataset_root": project_root / "data" / "gist",
        "dataset_name": "GIST",
        "case_suffix": "_CT",
        "modality": "CT",
        "expected_samples": 246
    },
    {
        "name": "LIPO",
        "category": "lipo",
        "ct_name": "ct_LIPO_roi",
        "dataset_root": project_root / "data" / "lipo",
        "dataset_name": "Lipo",
        "case_suffix": "_MR",
        "modality": "MR",
        "expected_samples": 115
    },
    {
        "name": "DESMOID",
        "category": "desmoid",
        "ct_name": "ct_DESMOID_roi",
        "dataset_root": project_root / "data" / "desmoid",
        "dataset_name": "Desmoid",
        "case_suffix": "_MR",  # FIXED: Desmoid folders are named _MR not _CT
        "modality": "MR",      # FIXED: Actually MR modality based on folder naming
        "expected_samples": 203
    },
    {
        "name": "CRLM",
        "category": "crlm",
        "ct_name": "ct_CRLM_roi",
        "dataset_root": project_root / "data" / "crlm",
        "dataset_name": "CRLM",
        "case_suffix": "_CT",
        "modality": "CT",
        "expected_samples": 77
    }
]

print(f"✅ Setup complete")
print(f"   Project root: {project_root}")
print(f"   SAM-Med3D root: {SAM3D_ROOT}")
print(f"   Device: {DEVICE}")
print(f"   Datasets to process: {[d['name'] for d in DATASETS]}")
print(f"   Total expected samples: {sum(d['expected_samples'] for d in DATASETS)}")

✅ Setup complete
   Project root: c:\Users\cahel\Desktop\Med3Tab-PFN
   SAM-Med3D root: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main
   Device: cpu
   Datasets to process: ['GIST', 'LIPO', 'DESMOID', 'CRLM']
   Total expected samples: 641


## 2. Preprocessing with ROI Cropping

Preprocess images for **all datasets** using `prepare_for_sam3d_roi_cropped` which:
- Finds the tumor bounding box from the segmentation mask
- Crops around it with a margin for context
- Resizes/pads to 128³ (centered)

This replaces the custom `crop_to_roi_simple` function with the existing implementation from `med3pipe.data.prepare`.

In [2]:
# Store results for all datasets
all_results = {}

# Process each dataset
for dataset_config in DATASETS:
    print(f"\n{'='*70}")
    print(f"PROCESSING DATASET: {dataset_config['name']}")
    print(f"{'='*70}\n")
    
    # Prepare ROI-cropped data using existing function
    print(f"Step 1: Preparing ROI-cropped data for {dataset_config['name']}...")
    
    n_prepared, paths = prepare_for_sam3d_roi_cropped(
        dataset_root=dataset_config['dataset_root'],
        sam3d_root=SAM3D_ROOT,
        category=dataset_config['category'],
        ct_name=dataset_config['ct_name'],
        target_size=TARGET_SIZE,
        margin=ROI_MARGIN,
        max_cases=None  # Process all cases
    )
    
    print(f"✅ Prepared {n_prepared} ROI-cropped cases for {dataset_config['name']}")
    print(f"   Output: {paths.train_root}")
    
    # Store paths for later use
    dataset_config['paths'] = paths

print(f"\n{'='*70}")
print("ALL DATASETS PREPROCESSED")
print(f"{'='*70}")


PROCESSING DATASET: GIST

Step 1: Preparing ROI-cropped data for GIST...
Prepared 25 ROI-cropped cases ...
Prepared 25 ROI-cropped cases ...
Prepared 50 ROI-cropped cases ...
Prepared 50 ROI-cropped cases ...
Prepared 75 ROI-cropped cases ...
Prepared 75 ROI-cropped cases ...
Prepared 100 ROI-cropped cases ...
Prepared 100 ROI-cropped cases ...
Prepared 125 ROI-cropped cases ...
Prepared 125 ROI-cropped cases ...
Prepared 150 ROI-cropped cases ...
Prepared 150 ROI-cropped cases ...
Prepared 175 ROI-cropped cases ...
Prepared 175 ROI-cropped cases ...
Prepared 200 ROI-cropped cases ...
Prepared 200 ROI-cropped cases ...
Prepared 225 ROI-cropped cases ...
Prepared 225 ROI-cropped cases ...
Done. Prepared 246 ROI-cropped cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST_roi
✅ Prepared 246 ROI-cropped cases for GIST
   Output: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST_roi

PROCESSING DATASET: LI

## 3. Load SAM-Med3D Model

Load the pre-trained SAM-Med3D model using `build_sam3d_model` from `med3pipe.sam.core`.
This function supports both medim and legacy loading methods.

In [3]:
# Check if checkpoint exists
if not CHECKPOINT_PATH.exists():
    print(f"⚠️  Checkpoint not found: {CHECKPOINT_PATH}")
    print("   Download from: https://huggingface.co/blueyo0/SAM-Med3D/resolve/main/sam_med3d_turbo.pth")
else:
    size_mb = CHECKPOINT_PATH.stat().st_size / (1024*1024)
    print(f"✅ Checkpoint found: {CHECKPOINT_PATH} ({size_mb:.1f} MB)")

# Load SAM-Med3D model using existing function
print("\n📦 Loading SAM-Med3D model...")
sam_model = build_sam3d_model(
    sam3d_root=SAM3D_ROOT,
    model_type="vit_b_ori",
    checkpoint=CHECKPOINT_PATH,
    device=DEVICE,
    eval_mode=True,
    use_medim=True
)

print(f"✅ SAM-Med3D model loaded successfully!")
print(f"   Model device: {next(sam_model.parameters()).device}")

✅ Checkpoint found: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth (383.5 MB)

📦 Loading SAM-Med3D model...
Loading SAM-Med3D model via medim from: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth
creating model SAM-Med3D
try to load pretrained weights from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth
✅ SAM-Med3D model loaded successfully via medim!
✅ SAM-Med3D model loaded successfully!
   Model device: cpu
✅ SAM-Med3D model loaded successfully via medim!
✅ SAM-Med3D model loaded successfully!
   Model device: cpu


## 4. Extract Embeddings from Encoder

Extract embeddings for all datasets using `extract_embeddings_train_val` from `med3pipe.sam.core`.
This function handles the preprocessing pipeline and saves embeddings to disk.

In [4]:
# Extract embeddings for all datasets using existing function
for dataset_config in DATASETS:
    print(f"\n{'='*70}")
    print(f"EXTRACTING EMBEDDINGS: {dataset_config['name']}")
    print(f"{'='*70}\n")
    
    paths = dataset_config['paths']
    
    # Get or create feature directories
    feature_dirs = default_feature_dirs(
        sam3d_root=SAM3D_ROOT,
        category=dataset_config['category'],
        ct_name=dataset_config['ct_name']
    )
    
    # Extract embeddings for train set (no validation split for now)
    print(f"Extracting embeddings from {paths.images_tr}...")
    feature_dirs = extract_embeddings_train_val(
        paths=paths,
        model=sam_model,
        sam3d_root=SAM3D_ROOT,
        img_size=TARGET_SIZE,
        feature_dirs=feature_dirs,
        device=DEVICE,
        skip_existing=True
    )
    
    print(f"✅ Extracted embeddings for {dataset_config['name']}")
    print(f"   Train features: {feature_dirs.train_dir}")
    
    # Store feature dirs for later use
    dataset_config['feature_dirs'] = feature_dirs

print(f"\n{'='*70}")
print("ALL EMBEDDINGS EXTRACTED")
print(f"{'='*70}")


EXTRACTING EMBEDDINGS: GIST

Extracting embeddings from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST_roi\imagesTr...
To extract: 246 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST_roi\imagesTr
Extracted 25 embeddings ...
Extracted 25 embeddings ...
Extracted 50 embeddings ...
Extracted 50 embeddings ...
Extracted 75 embeddings ...
Extracted 75 embeddings ...
Extracted 100 embeddings ...
Extracted 100 embeddings ...
Extracted 125 embeddings ...
Extracted 125 embeddings ...
Extracted 150 embeddings ...
Extracted 150 embeddings ...
Extracted 175 embeddings ...
Extracted 175 embeddings ...
Extracted 200 embeddings ...
Extracted 200 embeddings ...
Extracted 225 embeddings ...
Extracted 225 embeddings ...
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST_roi_train
To extract: 0 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-mai

## 5. Pool Embeddings to Feature Vectors

Apply Global Average Pooling (GAP) using `load_pooled_features` from `med3pipe.sam.core`.
This converts 3D embeddings to 1D feature vectors for classification.

In [5]:
# Pool embeddings for all datasets using existing function
for dataset_config in DATASETS:
    print(f"\n{'='*70}")
    print(f"POOLING EMBEDDINGS: {dataset_config['name']}")
    print(f"{'='*70}\n")
    
    feature_dirs = dataset_config['feature_dirs']
    paths = dataset_config['paths']
    
    # Load and pool features using existing function (Global Average Pooling)
    X, case_ids = load_pooled_features(
        feat_dir=feature_dirs.train_dir,
        label_dir=None,  # Not needed for GAP
        pre_transform=None
    )
    
    print(f"✅ Pooled embeddings for {dataset_config['name']}")
    print(f"   Feature matrix shape: {X.shape}")
    print(f"   Number of cases: {len(case_ids)}")
    print(f"   Sample case IDs: {case_ids[:5]}")
    
    # Store features and IDs for later use
    dataset_config['X'] = X
    dataset_config['case_ids'] = case_ids

print(f"\n{'='*70}")
print("ALL EMBEDDINGS POOLED")
print(f"{'='*70}")


POOLING EMBEDDINGS: GIST

✅ Pooled embeddings for GIST
   Feature matrix shape: (246, 384)
   Number of cases: 246
   Sample case IDs: ['GIST-001_CT.nii', 'GIST-002_CT.nii', 'GIST-003_CT.nii', 'GIST-004_CT.nii', 'GIST-005_CT.nii']

POOLING EMBEDDINGS: LIPO

✅ Pooled embeddings for GIST
   Feature matrix shape: (246, 384)
   Number of cases: 246
   Sample case IDs: ['GIST-001_CT.nii', 'GIST-002_CT.nii', 'GIST-003_CT.nii', 'GIST-004_CT.nii', 'GIST-005_CT.nii']

POOLING EMBEDDINGS: LIPO

✅ Pooled embeddings for LIPO
   Feature matrix shape: (115, 384)
   Number of cases: 115
   Sample case IDs: ['Lipo-001_MR.nii', 'Lipo-002_MR.nii', 'Lipo-003_MR.nii', 'Lipo-004_MR.nii', 'Lipo-005_MR.nii']

POOLING EMBEDDINGS: DESMOID

✅ Pooled embeddings for LIPO
   Feature matrix shape: (115, 384)
   Number of cases: 115
   Sample case IDs: ['Lipo-001_MR.nii', 'Lipo-002_MR.nii', 'Lipo-003_MR.nii', 'Lipo-004_MR.nii', 'Lipo-005_MR.nii']

POOLING EMBEDDINGS: DESMOID

✅ Pooled embeddings for DESMOID
   Feat

## 6. Load Labels from sheet.csv

Load ground truth labels using `load_labels_from_sheet` and `build_y` from `med3pipe.sam.core`.
These functions handle label loading and alignment with case IDs for all datasets.

In [6]:
# Load labels for all datasets using existing function
sheet_path = project_root / "sheet.csv"

for dataset_config in DATASETS:
    print(f"\n{'='*70}")
    print(f"LOADING LABELS: {dataset_config['name']}")
    print(f"{'='*70}\n")
    
    # Load labels from sheet using existing function
    df, label_map = load_labels_from_sheet(
        sheet_csv=sheet_path,
        dataset_name=dataset_config['dataset_name'],
        subject_col="Subject",
        label_col="Diagnosis_binary",
        case_suffix=dataset_config['case_suffix']
    )
    
    print(f"✅ Loaded {len(label_map)} labels from sheet.csv for {dataset_config['name']}")
    
    # Build y array using existing function
    case_ids = dataset_config['case_ids']
    y, missing = build_y(case_ids, label_map)
    
    if missing:
        print(f"⚠️  Warning: {len(missing)} cases without labels: {missing[:5]}")
    
    print(f"   Aligned {len(y)} cases with labels")
    print(f"   Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
    
    # Store labels for later use
    dataset_config['y'] = y
    dataset_config['label_map'] = label_map

print(f"\n{'='*70}")
print("ALL LABELS LOADED")
print(f"{'='*70}")


LOADING LABELS: GIST

✅ Loaded 246 labels from sheet.csv for GIST
   Aligned 246 cases with labels
   Class distribution: {np.int64(0): np.int64(121), np.int64(1): np.int64(125)}

LOADING LABELS: LIPO

✅ Loaded 115 labels from sheet.csv for LIPO
   Aligned 115 cases with labels
   Class distribution: {np.int64(0): np.int64(58), np.int64(1): np.int64(57)}

LOADING LABELS: DESMOID

✅ Loaded 203 labels from sheet.csv for DESMOID
   Aligned 203 cases with labels
   Class distribution: {np.int64(0): np.int64(131), np.int64(1): np.int64(72)}

LOADING LABELS: CRLM

✅ Loaded 77 labels from sheet.csv for CRLM
   Aligned 77 cases with labels
   Class distribution: {np.int64(0): np.int64(40), np.int64(1): np.int64(37)}

ALL LABELS LOADED


## 7. Classification with TabPFN

Perform dimensionality reduction (PCA) and classification with TabPFN for **all datasets**.
Results are stored for the final performance summary table.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Add TabPFN to path if needed
tabpfn_src = project_root / "TabPFN-main" / "TabPFN-main" / "src"
if tabpfn_src.exists() and str(tabpfn_src) not in sys.path:
    sys.path.insert(0, str(tabpfn_src))

from tabpfn.classifier import TabPFNClassifier

# Train and evaluate on each dataset
for dataset_config in DATASETS:
    print(f"\n{'='*70}")
    print(f"CLASSIFICATION: {dataset_config['name']}")
    print(f"{'='*70}\n")
    
    X = dataset_config['X']
    y = dataset_config['y']
    
    # Check if we have enough samples
    n_samples = len(y)
    n_classes = len(np.unique(y))
    
    print(f"Dataset: {n_samples} samples, {n_classes} classes")
    print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
    
    if n_samples < 10:
        print(f"\n⚠️  WARNING: Only {n_samples} samples available.")
        print("   Using simple split without stratification for demonstration.\n")
        test_size = max(2, n_samples // 5)
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=test_size, random_state=42, shuffle=True
        )
    else:
        # Normal stratified split for larger datasets
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
    
    print(f"Train set: {len(y_train)} samples")
    print(f"Val set: {len(y_val)} samples")
    
    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # PCA (TabPFN works best with ≤500 features)
    n_components = min(500, X_train_scaled.shape[0] - 1, X_train_scaled.shape[1])
    pca = PCA(n_components=n_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_val_pca = pca.transform(X_val_scaled)
    
    print(f"\n✅ Preprocessing complete")
    print(f"   PCA components: {n_components}")
    print(f"   Explained variance: {pca.explained_variance_ratio_.sum():.3f}")
    
    # Train TabPFN
    print("\n📊 Training TabPFN...")
    clf = TabPFNClassifier(device="cuda" if torch.cuda.is_available() else "cpu")
    clf.fit(X_train_pca, y_train)
    
    # Evaluate
    y_pred = clf.predict(X_val_pca)
    y_proba = clf.predict_proba(X_val_pca)
    
    accuracy = accuracy_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_proba[:, 1]) if len(np.unique(y)) == 2 else None
    
    print(f"\n✅ TabPFN Classification Results for {dataset_config['name']}")
    print(f"   Accuracy: {accuracy:.3f}")
    if roc_auc:
        print(f"   ROC AUC: {roc_auc:.3f}")
    print(f"\nClassification Report:")
    print(classification_report(y_val, y_pred))
    
    # Store results for summary table
    dataset_config['results'] = {
        'accuracy': accuracy,
        'roc_auc': roc_auc,
        'n_train': len(y_train),
        'n_val': len(y_val),
        'n_components': n_components,
        'explained_variance': pca.explained_variance_ratio_.sum()
    }

print(f"\n{'='*70}")
print("ALL DATASETS CLASSIFIED")
print(f"{'='*70}")

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)



CLASSIFICATION: GIST

Dataset: 246 samples, 2 classes
Class distribution: {np.int64(0): np.int64(121), np.int64(1): np.int64(125)}
Train set: 196 samples
Val set: 50 samples

✅ Preprocessing complete
   PCA components: 195
   Explained variance: 1.000

📊 Training TabPFN...

✅ TabPFN Classification Results for GIST
   Accuracy: 0.620
   ROC AUC: 0.696

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.48      0.56        25
           1       0.59      0.76      0.67        25

    accuracy                           0.62        50
   macro avg       0.63      0.62      0.61        50
weighted avg       0.63      0.62      0.61        50


CLASSIFICATION: LIPO

Dataset: 115 samples, 2 classes
Class distribution: {np.int64(0): np.int64(58), np.int64(1): np.int64(57)}
Train set: 92 samples
Val set: 23 samples

✅ Preprocessing complete
   PCA components: 91
   Explained variance: 1.000

📊 Training TabPFN...

✅ TabPFN Classification

## 8. Performance Summary Table

Create a comprehensive summary table comparing performance across all datasets.

In [8]:
import pandas as pd

# Create performance summary table
print("\n" + "="*70)
print("PERFORMANCE SUMMARY TABLE")
print("="*70 + "\n")

summary_data = []
for dataset_config in DATASETS:
    results = dataset_config['results']
    summary_data.append({
        'Dataset': dataset_config['name'],
        'Modality': dataset_config.get('modality', 'CT'),
        'N_Train': results['n_train'],
        'N_Val': results['n_val'],
        'N_Total': results['n_train'] + results['n_val'],
        'PCA_Components': results['n_components'],
        'Expl_Var': f"{results['explained_variance']:.3f}",
        'Accuracy': f"{results['accuracy']:.3f}",
        'ROC_AUC': f"{results['roc_auc']:.3f}" if results['roc_auc'] is not None else 'N/A'
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Display as styled DataFrame for better visualization
display(summary_df)

print("\n" + "="*70)
print("PIPELINE SUMMARY")
print("="*70)
print("""
This notebook demonstrates an end-to-end pipeline for 4 tumor datasets:
- GIST (CT) - Gastrointestinal Stromal Tumors
- LIPO (MR) - Liposarcoma
- DESMOID (CT) - Desmoid Tumors  
- CRLM (CT) - Colorectal Liver Metastases

Pipeline steps:
1. ✅ **ROI-cropped preprocessing** - Tumor-centered volumes using prepare_for_sam3d_roi_cropped
2. ✅ **SAM-Med3D loading** - Via build_sam3d_model from med3pipe.sam.core
3. ✅ **Embedding extraction** - Using extract_embeddings_train_val from med3pipe.sam.core
4. ✅ **Feature pooling** - Global Average Pooling via load_pooled_features
5. ✅ **Label loading** - Using load_labels_from_sheet and build_y from med3pipe.sam.core
6. ✅ **TabPFN classification** - With PCA preprocessing and performance evaluation

Note: LIPO uses MRI data, while SAM-Med3D was trained on CT. Lower performance on LIPO is expected.
""")


PERFORMANCE SUMMARY TABLE

Dataset Modality  N_Train  N_Val  N_Total  PCA_Components Expl_Var Accuracy ROC_AUC
   GIST       CT      196     50      246             195    1.000    0.620   0.696
   LIPO       MR       92     23      115              91    1.000    0.565   0.500
DESMOID       MR      162     41      203             161    1.000    0.659   0.746
   CRLM       CT       61     16       77              60    1.000    0.500   0.531


,Dataset,Modality,N_Train,N_Val,N_Total,PCA_Components,Expl_Var,Accuracy,ROC_AUC
0,GIST,CT,196,50,246,195,1.000,0.620,0.696
1,LIPO,MR,92,23,115,91,1.000,0.565,0.500
2,DESMOID,MR,162,41,203,161,1.000,0.659,0.746
3,CRLM,CT,61,16,77,60,1.000,0.500,0.531



PIPELINE SUMMARY

This notebook demonstrates an end-to-end pipeline for 4 tumor datasets:
- GIST (CT) - Gastrointestinal Stromal Tumors
- LIPO (MR) - Liposarcoma
- DESMOID (CT) - Desmoid Tumors  
- CRLM (CT) - Colorectal Liver Metastases

Pipeline steps:
1. ✅ **ROI-cropped preprocessing** - Tumor-centered volumes using prepare_for_sam3d_roi_cropped
2. ✅ **SAM-Med3D loading** - Via build_sam3d_model from med3pipe.sam.core
3. ✅ **Embedding extraction** - Using extract_embeddings_train_val from med3pipe.sam.core
4. ✅ **Feature pooling** - Global Average Pooling via load_pooled_features
5. ✅ **Label loading** - Using load_labels_from_sheet and build_y from med3pipe.sam.core
6. ✅ **TabPFN classification** - With PCA preprocessing and performance evaluation

Note: LIPO uses MRI data, while SAM-Med3D was trained on CT. Lower performance on LIPO is expected.

